In [1]:
# cell 1
!pip -q install -U torch transformers peft bitsandbytes datasets accelerate
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import os

from google.colab import drive
import os
import shutil

print("Mounting Google Drive...")
drive.mount('/content/drive')

Mounting Google Drive...
Mounted at /content/drive


In [2]:
# cell 2
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
NEW_MODEL_NAME = "HealthLens-Pro-Tuned"

# (Quantization)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Loading Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [3]:
# cell 3
model = prepare_model_for_kbit_training(model)

# LoRA
peft_config = LoraConfig(
    lora_alpha=128,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 50,462,720 || all params: 1,150,511,104 || trainable%: 4.3861


In [4]:
# cell 4
print("Loading Dataset...")
dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")

# 20,000
dataset = dataset.shuffle(seed=42).select(range(20000))

dataset = dataset.train_test_split(test_size=0.1)

def format_instruction(sample):
    return f"### Instruction:\nYou are HealthLens, a medical assistant. Answer the user's question safely.\n\n### User:\n{sample['instruction']} {sample['input']}\n\n### Assistant:\n{sample['output']}<|endoftext|>"

def tokenize_function(examples):
    texts = [format_instruction(dict(zip(examples, t))) for t in zip(*examples.values())]
    tokenized = tokenizer(texts, truncation=True, max_length=512, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("Tokenizing data...")
tokenized_train = dataset["train"].map(tokenize_function, batched=True, remove_columns=dataset["train"].column_names)
tokenized_eval = dataset["test"].map(tokenize_function, batched=True, remove_columns=dataset["test"].column_names)

Loading Dataset...


README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…):   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

Tokenizing data...


Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [5]:
# cell 5
training_args = TrainingArguments(
    output_dir="./healthlens_results",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    eval_strategy="no",
    save_strategy="no",
    warmup_ratio=0.03,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

In [6]:
# cell 6
torch.cuda.empty_cache()
print("Starting Training...")
trainer.train()

trainer.model.save_pretrained(NEW_MODEL_NAME)
tokenizer.save_pretrained(NEW_MODEL_NAME)
print("Training Complete and Model Saved!")

Starting Training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,2.578200
40,2.139100
60,2.049400
80,2.022000
100,2.026000
120,1.986400
140,1.992200
160,1.944800
180,1.938300
200,1.931000


Training Complete and Model Saved!


In [7]:
# cell 7
import torch

def ask_healthlens(question):
    model.eval()

    prompt = f"### Instruction:\nYou are HealthLens, a medical assistant. Answer the user's question safely.\n\n### User:\n{question}\n\n### Assistant:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "### Assistant:" in response:
        final_answer = response.split("### Assistant:")[-1].strip()
    else:
        final_answer = response

    return final_answer

print("="*40)
print("Test 1: Flu Symptoms")
print(ask_healthlens("I have a fever, body aches, and a runny nose. What could it be?"))

print("="*40)
print("Test 2: Emergency")
print(ask_healthlens("My chest hurts and I can't breathe well."))

Test 1: Flu Symptoms
Hi Dear, Welcome to Chat Doctor. Understanding your concern. As per your query you have symptoms of fever with bodyache along with running nose which is due to upper respiratory tract infection, allergic rhinitis or sinusitis. Need not worry about it. I would suggest you to visit ENT specialist once and get it examined and start treatment after proper diagnosis. You should take antihistamine tablet like Levocetrizine along with decongestant Chlorpheniramine male ate 12 hourly for five days. Do warm saline gargles twice daily. Take steam inhalation several times a day. For now give cold soft foods so that your throat will open up more easily. Avoid stress as this can worsen condition further. Hope your concern has been resolved. Get Well Soon. Best Wishes,<|endoftext|><
Test 2: Emergency
Thanks for your question on Chat Doctor. In my opinion you should consult pulmonologist first because as per your history of chest pain there is possibility of bronchitis in your ca

In [8]:
# cell 8
drive_path = f"/content/drive/MyDrive/{NEW_MODEL_NAME}"

if not os.path.exists(drive_path):
    os.makedirs(drive_path)

print(f"Copying model files to {drive_path}...")
!cp -r {NEW_MODEL_NAME}/* "{drive_path}"

print(f"✅ Backup Complete! Check your Google Drive folder: {NEW_MODEL_NAME}")

Copying model files to /content/drive/MyDrive/HealthLens-Pro-Tuned...
✅ Backup Complete! Check your Google Drive folder: HealthLens-Pro-Tuned
